# All feature extraction: every ICA feature on any cell and cycle

This notebook draws each feature the pipeline extracts on the real discharge dQ/dV curve of
the cell and cycle you choose. It also shows the **170 rows that the cleaning step removes**
from the feature table, and why each one is removed.

**The drawings are the extractor's own geometry.** Values come from
`src/features._dominant_peak_features`, the function that wrote
`results/features_expanded.parquet` (ENG-05). The landmarks (prominence bases, half-prominence
crossings, integration windows, flank points) come from the same scipy calls. Every drawn
cycle is compared with the cached table, and §3.4 checks a random sample. History features
come from `src/features_temporal.add_temporal_features`, as in ENG-14.

| § | contents |
|---|---|
| 1 | setup and data (about 25 s: loads the three pickles, keeps only what the plots need) |
| 2 | what each feature is |
| 3 | **viewer**: pick a battery and a cycle; see all nine features, or one feature with its trajectory and history features |
| 4 | **the 170 dropped rows**: what they are, one example of each kind, and a browser for all 170 |
| 5 | **published vs recomputed dQ/dV**: the curve rebuilt from the raw samples (`src/dqdv.py`), with a smoothing slider and the features measured on both curves |
| 6 | **odd cycles**: noisy cells, published curves that disagree with their own data, abnormal recordings, a browser for all of them, which cells look bad, and what removing them changes |
| 7 | notes |

**Using it.** Run all cells (in VS Code: *Run All* with the `.venv` kernel), then use the
controls in §3.6, §4.4, §5.3 and §6.3. A plot redraws when you release a slider or press ◀ / ▶.
Each view is also a plain function (`show_cycle`, `show_feature`, `show_dropped`,
`show_recomputed`) that returns a matplotlib figure, so any view can be saved with
`fig.savefig(...)`.

**170 rows, not 170 cells.** `clean_feature_table` keeps 100,331 of the 100,501 cycle-rows.
The 170 it drops come from all 124 cells, because every cell loses its first one or two cycles.
Only 42 of the 170 are extraction failures. The other 128 have normal features and are dropped
because of their SoH label. 124 of those 128 trace back to a one-cycle offset in how the label is
looked up (§4.2).

## 1. Setup and data

In [ ]:
import gc
import json
import os
import sys
import time
from pathlib import Path

import ipywidgets as widgets
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from scipy.signal import find_peaks, peak_prominences, peak_widths

# --- run from the repo root (the parent of this notebooks/ folder) so src/ imports resolve ---
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":  # the kernel starts in the notebook's folder
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src" / "severson_loader.py").exists(), "open this notebook from the notebooks/ folder"
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_prep import SOH_MAX, SOH_MIN, clean_feature_table
from src.dqdv import (I_DISCHARGE_MAX, SMOOTH_POLYORDER, SMOOTH_WINDOW, compare_with_published,
                      dqdv_from_raw)
from src.features_temporal import add_temporal_features
from src.features import (AREA_WINDOWS, FEATURE_COLS, _dominant_peak_features,
                          _interp_abs_dqdv)
from src.peak_extraction import (DISTANCE, DV, PROMINENCE, ROI_MASK, V_GRID, V_GRID_POINTS,
                                 V_ROI)
from src.severson_loader import load_clean_severson

FEATURES_PARQUET = REPO_ROOT / "results" / "features_expanded.parquet"
NB_DIR = REPO_ROOT / "notebooks"  # dropped_rows.csv and views/ are written here
FLANK = 0.025  # src.features measures both flank slopes over 25 mV
EDGE_V = V_ROI[-1] + 0.05  # y-limits ignore the 2.00-2.05 V end, where one-point spikes live (§6)

# Colours: the curve is grey ink and the feature on show is one accent (C1). C1-C3 are the
# first three slots of a palette checked for colour-blind separation. CRITICAL is a status
# colour, used only for rows the cleaning step drops.
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
CRITICAL = "#d03b3b"
BLUES = LinearSegmentedColormap.from_list(
    "blues", ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281", "#0d366b"])

plt.rcParams.update({
    "figure.dpi": 90, "savefig.dpi": 200,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": ["Segoe UI", "DejaVu Sans"],  # a list, so missing glyphs fall back
    "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9, "legend.fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "legend.frameon": False, "axes.unicode_minus": False,
})
print("repo root:", REPO_ROOT)

### 1.1 Load

Nothing is written back to `src/` or `results/`. Inputs:
- `results/features_expanded.parquet`, the ENG-05 table with every cycle-row, failures included;
- its cleaned version (`clean_feature_table`) and the ENG-14 history features computed on it;
- the three Severson pickles through `load_clean_severson()`, which needs about 8 GB while
  loading. From them the notebook keeps every cycle's 1000-point dQ/dV, the summary capacity `QD`,
  the maximum of every cycle's raw `Qd` trace, every cycle's raw discharge samples (V and Qd while
  I < −0.5 A, for the recomputed curves of §5), a few checks per cycle for §6, and all raw signals
  of the dropped cycles and their neighbours (about 1.4 GB in all). The rest is freed.

In [ ]:
t0 = time.time()
if not FEATURES_PARQUET.exists():
    raise FileNotFoundError(f"{FEATURES_PARQUET} is missing: run stages/s02_extract_features.py (~90 s)")
feat = pd.read_parquet(FEATURES_PARQUET)
clean, _ = clean_feature_table(feat, verbose=True)
hist, HISTORY_COLS = add_temporal_features(clean)  # ENG-14: 54 history columns

# Why each row leaves the table, in the order clean_feature_table applies its filters.
soh = feat["SOH"]
reason = pd.Series("kept", index=feat.index, dtype=object)
reason[soh > SOH_MAX] = "soh_high"
reason[soh <= SOH_MIN] = "soh_low"
reason[soh.isna()] = "soh_nan"
failed = feat["status"] != "ok"
reason[failed] = feat.loc[failed, "status"]
feat["reason"] = reason
assert (feat["reason"] == "kept").sum() == len(clean)
print(f"feature table: {len(feat):,} rows from {feat['cell_id'].nunique()} cells "
      f"({time.time() - t0:.1f} s)")

In [ ]:
# Thresholds of the per-cycle checks in section 6.
BLOCK_RATIO, BLOCK_POINTS = 1.5, 5  # published dQ/dV > 1.5 x its Qdlin slope for >= 7.5 mV
LONG_RECORDING_H = 2.0  # a normal cycle is recorded in about one hour


def _longest_run(mask):
    """(length, start index) of the longest run of True values, (0, -1) when there is none."""
    edges = np.flatnonzero(np.diff(np.concatenate(([0], mask.astype(np.int8), [0]))))
    if edges.size == 0:
        return 0, -1
    starts, ends = edges[::2], edges[1::2]
    i = int(np.argmax(ends - starts))
    return int(ends[i] - starts[i]), int(starts[i])


def _read_cycle(cyc, d):
    """The cycle's discharge samples, selected exactly as src.dqdv.recompute_dqdv selects them
    (None where it would return all-NaN), and the section-6 checks."""
    I = np.asarray(cyc["I"], dtype=float)
    t = np.asarray(cyc["t"], dtype=float)
    checks = {"hours": float(t.max()) / 60 if t.size >= 10 else np.nan}
    on = I < I_DISCHARGE_MAX
    samples = None
    if I.size >= 10 and on.sum() >= 10:
        samples = (np.asarray(cyc["V"], dtype=float)[on], np.asarray(cyc["Qd"], dtype=float)[on])
        checks["discharge_pieces"] = int(np.count_nonzero(np.diff(on.astype(np.int8)) == 1) + on[0])
        checks["qd_start"] = float(samples[1][0])
    qdlin = np.asarray(cyc["Qdlin"], dtype=float)
    if d.size == V_GRID_POINTS and qdlin.size == V_GRID_POINTS:
        pub = -d[ROI_MASK]
        slope = -np.gradient(qdlin, V_GRID)[ROI_MASK]  # dQ/dV implied by the dataset's own Q(V)
        run, i = _longest_run((slope > 0.5) & (pub > BLOCK_RATIO * slope))
        checks["block_points"] = run
        if run:
            checks["block_lo_v"], checks["block_hi_v"] = float(V_ROI[i + run - 1]), float(V_ROI[i])
            checks["block_ratio"] = float(np.max(pub[i:i + run] / slope[i:i + run]))
        checks["edge_spike"] = bool(np.argmax(pub) >= pub.size - 5)
    return samples, checks


t0 = time.time()
cells = load_clean_severson(verbose=False)  # ~8 GB until the `del` below
dropped_pairs = feat.loc[feat["reason"] != "kept", ["cell_id", "cycle"]].itertuples(index=False)
keep_raw = {(c, k + d) for c, k in dropped_pairs for d in (-2, -1, 0, 1)}

CURVES, MAXQD, QD, QD_REF, SUMMARY_CYCLE, RAW, DISCHARGE = {}, {}, {}, {}, {}, {}, {}
scan = []
for cid, cell in cells.items():
    n = len(cell["cycles"])
    curves = np.full((n, V_GRID_POINTS), np.nan)  # NaN row = no 1000-point curve
    maxqd = np.full(n, np.nan)
    DISCHARGE[cid] = [None] * n
    for k in range(n):
        cyc = cell["cycles"][str(k)]
        d = np.asarray(cyc["dQdV"], dtype=float)
        if d.size == V_GRID_POINTS:
            curves[k] = d
        q = np.asarray(cyc["Qd"], dtype=float)
        if q.size > 2:
            maxqd[k] = q.max()
        if (cid, k) in keep_raw:
            RAW[(cid, k)] = {ch: np.array(v, dtype=float) for ch, v in cyc.items()}
        DISCHARGE[cid][k], checks = _read_cycle(cyc, d)
        scan.append({"cell_id": cid, "cycle": k, **checks})
    CURVES[cid], MAXQD[cid] = curves, maxqd
    QD[cid] = np.asarray(cell["summary"]["QD"], dtype=float)
    SUMMARY_CYCLE[cid] = np.asarray(cell["summary"]["cycle"]).astype(int)
    QD_REF[cid] = float(QD[cid][QD[cid] > 0][0])  # the extractor's SoH reference
del cells
gc.collect()
SCAN = pd.DataFrame(scan).set_index(["cell_id", "cycle"]).sort_index()

# Each cycle's own capacity as an SoH value: summary row k belongs to cycle k (see 4.2).
own = np.full(len(feat), np.nan)
cycle_col = feat["cycle"].to_numpy()
for cid, idx in feat.groupby("cell_id").indices.items():
    own[idx] = QD[cid][cycle_col[idx]] / QD_REF[cid]
feat["SOH_own"] = own

FEAT = feat.set_index(["cell_id", "cycle"]).sort_index()
HIST = hist.set_index(["cell_id", "cycle"]).sort_index()
OK_POS = {cid: (g["cycle"].to_numpy(), g["position"].to_numpy())
          for cid, g in feat[feat["status"] == "ok"].sort_values("cycle").groupby("cell_id")}
CELL_IDS = sorted(CURVES, key=lambda c: (int(c[1]), int(c[3:])))
META = feat.groupby("cell_id").agg(policy=("charge_policy", "first"), n_cycles=("cycle", "size"))
DROPPED = feat[feat["reason"] != "kept"].reset_index(drop=True)

gb = (sum(a.nbytes for a in CURVES.values())
      + sum(v.nbytes + q.nbytes for s in DISCHARGE.values() for v, q in filter(None, s))) / 1e9
print(f"{len(CURVES)} cells, {sum(len(a) for a in CURVES.values()):,} cycles; curves and discharge "
      f"samples in memory {gb:.2f} GB; all raw signals kept for {len(RAW)} cycles "
      f"({time.time() - t0:.0f} s)")

## 2. The features

**Detection.** On every cycle the extractor takes the published discharge dQ/dV (1000 points,
3.5 V down to 2.0 V), keeps the ROI from 3.4 V down to 2.0 V (the top 100 mV holds the
constant-voltage hold), and runs `scipy.signal.find_peaks` on −dQ/dV. A peak needs a prominence of
at least 0.3 Ah/V and a 20-point (30 mV) separation from other peaks. On a cell's first successful
cycle the most prominent peak wins. After that, the detected peak nearest the previous cycle's
position wins (tracking), because about 17 % of cycles show more than one peak, mostly in the
2.0–2.5 V tail.

**The nine level features** (`features.FEATURE_COLS`), all read off that tracked peak:

| feature | what it measures | unit |
|---|---|---|
| `height` | −dQ/dV at the apex | Ah/V |
| `position` | voltage of the apex | V |
| `prominence` | rise of the apex above the higher of its two bases; scipy searches for the bases out to the ROI ends | Ah/V |
| `area` | ∫ \|dQ/dV\| dV over ±50 mV around the apex (trapezoid on the grid points inside) | Ah |
| `area35` | the same over ±35 mV | Ah |
| `area25` | the same over ±25 mV | Ah |
| `fwhm` | width of the peak at half its prominence (`scipy.signal.peak_widths`, `rel_height=0.5`) | V |
| `slope_left` | (height − \|dQ/dV\| at apex − 25 mV) / 25 mV, the lower-voltage flank | Ah/V² |
| `slope_right` | (height − \|dQ/dV\| at apex + 25 mV) / 25 mV, the higher-voltage flank | Ah/V² |

**The six history features** (ENG-14, `features_temporal.py`) of each level feature *f* at
cycle *k*, so 54 columns. They exist only on kept rows with at least 20 cycles of clean history in
the cell. Lags count cycle numbers on a per-cell grid, forward-filled over the few missing cycles.

| column | definition |
|---|---|
| `f_d10`, `f_d20` | *f*(k) − *f*(k−10), *f*(k) − *f*(k−20) |
| `f_slope10`, `f_slope20` | least-squares slope of *f* over the last 10 / 20 cycles (units per cycle) |
| `f_rm20` | mean of *f* over the last 20 cycles |
| `f_drift` | *f*(k) − *f*(first clean cycle of the cell) |

## 3. Viewer

### 3.1 Geometry

`geometry(cell, cycle)` recomputes one cycle exactly as the extractor did, with the tracking prior
the extractor had at that point: the position of the cell's latest earlier cycle with status
`ok`. It returns the nine values and every landmark needed to draw them.

In [ ]:
FEATURE_INFO = {  # name: (unit, value format, one-line description)
    "height": ("Ah/V", "{:.3f}", "−dQ/dV at the tracked peak"),
    "position": ("V", "{:.4f}", "voltage of the tracked peak"),
    "prominence": ("Ah/V", "{:.3f}", "rise above the higher of the two bases (whole ROI shown)"),
    "area": ("Ah", "{:.4f}", "∫|dQ/dV| dV over ±50 mV of the peak"),
    "area35": ("Ah", "{:.4f}", "∫|dQ/dV| dV over ±35 mV of the peak"),
    "area25": ("Ah", "{:.4f}", "∫|dQ/dV| dV over ±25 mV of the peak"),
    "fwhm": ("V", "{:.4f}", "width at half the prominence"),
    "slope_left": ("Ah/V²", "{:.1f}", "drop over 25 mV on the lower-voltage flank ÷ 25 mV"),
    "slope_right": ("Ah/V²", "{:.1f}", "drop over 25 mV on the higher-voltage flank ÷ 25 mV"),
}
GRID_ORDER = ["height", "position", "prominence", "area", "area35", "area25",
              "fwhm", "slope_left", "slope_right"]
assert sorted(GRID_ORDER) == sorted(FEATURE_COLS)
_ROI_IDX = np.arange(V_ROI.size)


def _idx_to_v(x):
    """Fractional ROI index -> volts (the ROI runs from 3.4 V down to 2.0 V)."""
    return float(np.interp(x, _ROI_IDX, V_ROI))


def prev_position(cid, k):
    """The tracking prior at cycle k: the position of the latest earlier 'ok' cycle."""
    cycles, positions = OK_POS[cid]
    i = int(np.searchsorted(cycles, k)) - 1
    return None if i < 0 else float(positions[i])


def geometry(cid, k):
    """Values and landmarks of the nine features at (cell, cycle). None when the cycle has
    no 1000-point curve or no peak passes the detection threshold."""
    curve = CURVES[cid][k]
    if np.isnan(curve).all():
        return None
    roi = curve[ROI_MASK]
    prev = prev_position(cid, k)
    vals = _dominant_peak_features(roi, prev)  # the extractor's own numbers
    if vals is None:
        return None
    neg = -roi
    peaks, props = find_peaks(neg, prominence=PROMINENCE, distance=DISTANCE)
    sel = (int(np.argmax(props["prominences"])) if prev is None
           else int(np.argmin(np.abs(V_ROI[peaks] - prev))))
    i = int(peaks[sel])
    _, half_y, left_ip, right_ip = peak_widths(neg, [i], rel_height=0.5)
    # scipy's "left" is the lower index, which is the HIGHER voltage on this descending grid
    lb, rb = int(props["left_bases"][sel]), int(props["right_bases"][sel])
    absd = np.abs(roi)
    vp = vals["position"]
    cached = FEAT.loc[(cid, k), FEATURE_COLS].to_numpy(dtype=float)
    return {
        "cid": cid, "k": k, "v": V_ROI, "neg": neg, "absd": absd, "prev": prev, "idx": i,
        **vals,
        "matches_table": bool(np.array_equal(cached, [vals[c] for c in FEATURE_COLS])),
        "other_peaks": [float(V_ROI[p]) for j, p in enumerate(peaks) if j != sel],
        "base_v": (_idx_to_v(rb), _idx_to_v(lb)),  # (lower-V base, higher-V base)
        "base_y": (float(neg[rb]), float(neg[lb])),
        "ref_level": float(neg[i] - props["prominences"][sel]),
        "half_y": float(half_y[0]),
        "half_v": (_idx_to_v(right_ip[0]), _idx_to_v(left_ip[0])),  # (lower V, higher V)
        "flank_y": (_interp_abs_dqdv(absd, vp - FLANK, V_ROI[::-1]),
                    _interp_abs_dqdv(absd, vp + FLANK, V_ROI[::-1])),
    }


def history_at(cid, k, f):
    """ENG-14 history features of f at (cell, cycle), with the forward-filled per-cycle grid
    they were computed on. status: 'dropped' (row not in the clean table), 'short' (fewer than
    20 cycles of history) or 'ok'."""
    if (cid, k) not in HIST.index:
        return {"status": "dropped"}
    rows = HIST.loc[cid]
    first, last = int(rows.index[0]), int(rows.index[-1])
    grid = rows[f].reindex(range(first, last + 1)).ffill()
    out = {"status": "ok" if rows.loc[k, "has_history"] else "short", "first": first,
           "grid": grid, "filled": ~grid.index.isin(rows.index), "f_k": float(rows.loc[k, f])}
    if out["status"] == "ok":
        out.update({kind: float(rows.loc[k, f"{f}_{kind}"])
                    for kind in ("d10", "d20", "slope10", "slope20", "rm20", "drift")})
    return out


g = geometry("b1c1", 100)
print({c: round(g[c], 4) for c in GRID_ORDER}, "| same as cached table:", g["matches_table"])

### 3.2 Drawing one feature

One function per feature. Each draws the cycle's −dQ/dV in grey and the feature's construction in
blue: the apex, the integration window, the half-prominence width, the flank secant, and so on.
The panels zoom on the peak, except `prominence`, whose bases lie far out in the ROI.

In [ ]:
def _dot(ax, x, y, color=C1, ms=6.5, z=5):
    ax.plot([x], [y], "o", ms=ms, color=color, mec=SURFACE, mew=1.5, zorder=z)


def _arrow(ax, start, end, color=C1, style="<->", lw=1.5):
    ax.annotate("", xy=end, xytext=start, zorder=4,
                arrowprops=dict(arrowstyle=style, color=color, lw=lw, shrinkA=0, shrinkB=0))


def _label(ax, x, y, s, ha="left", va="center", color=INK, size=8):
    ax.text(x, y, s, ha=ha, va=va, fontsize=size, color=color, zorder=6,
            bbox=dict(boxstyle="round,pad=0.15", fc=SURFACE, ec="none", alpha=0.85))


def _setup(ax, g, full=False, lo=None):
    """Grey curve of one cycle, zoomed on the peak (whole ROI when full=True)."""
    vp, h = g["position"], g["height"]
    if full:
        lo, hi = V_ROI[-1], V_ROI[0]
    else:
        lo = max(min(vp - 0.16, lo if lo is not None else np.inf), V_ROI[-1])
        hi = min(vp + 0.13, V_ROI[0])
    view = (g["v"] >= lo) & (g["v"] <= hi)
    bottom = min(0.0, float(g["neg"][view].min()))
    top = max(1.25 * h, 1.08 * float(g["neg"][view & (g["v"] >= EDGE_V)].max()))
    ax.axhline(0, color=AXIS, lw=0.8, zorder=1)
    ax.plot(g["v"], g["neg"], color=INK2, lw=1.3, zorder=2)
    ax.set_xlim(lo, hi)
    ax.set_ylim(bottom - 0.03 * top, top)


def draw_height(ax, g):
    _setup(ax, g)
    ax.vlines(g["position"], 0, g["height"], color=C1, lw=2, zorder=3)
    _dot(ax, g["position"], g["height"])


def draw_position(ax, g):
    _setup(ax, g)
    vp = g["position"]
    ax.vlines(vp, ax.get_ylim()[0], g["height"], color=C1, lw=1.2, linestyles=(0, (3, 2)),
              zorder=3)
    ax.plot([vp], [0], marker="^", ms=9, color=C1, mec=SURFACE, mew=1.2, zorder=5,
            clip_on=False)
    _dot(ax, vp, g["height"])
    _label(ax, vp + 0.005, 0.08 * g["height"], f"{vp:.4f} V")
    lines = ["first cycle: the most prominent peak" if g["prev"] is None
             else f"tracking: nearest peak to the last ok cycle ({g['prev']:.4f} V)"]
    if g["other_peaks"]:
        shown = ", ".join(f"{v:.2f}" for v in g["other_peaks"][:4])
        more = "…" if len(g["other_peaks"]) > 4 else ""
        lines.append(f"other peaks found and ignored: {shown}{more} V")
    ax.text(0.02, 0.97, "\n".join(lines), transform=ax.transAxes, va="top", fontsize=7.3,
            color=INK2, zorder=6)


def draw_prominence(ax, g):
    _setup(ax, g, full=True)
    vp, top, ref = g["position"], g["neg"][g["idx"]], g["ref_level"]
    for v, y in zip(g["base_v"], g["base_y"]):
        ax.plot([v], [y], "^", ms=7, color=C1, mec=SURFACE, mew=1.2, zorder=5)
    v_ref = g["base_v"][int(np.argmax(g["base_y"]))]  # the higher base sets the reference
    ax.hlines(ref, min(v_ref, vp), max(v_ref, vp), color=C1, lw=1, linestyles=(0, (3, 2)),
              zorder=3)
    _arrow(ax, (vp, ref), (vp, top))
    _dot(ax, vp, top)
    side = ("right" if v_ref > V_ROI[0] - 0.15 else
            "left" if v_ref < V_ROI[-1] + 0.15 else "center")
    _label(ax, v_ref, ref + 0.13 * g["height"], "higher base", ha=side, va="bottom",
           color=INK2, size=7.5)


def draw_area(ax, g, name="area"):
    _setup(ax, g)
    hw, vp, h = AREA_WINDOWS[name], g["position"], g["height"]
    m = (g["v"] >= vp - hw) & (g["v"] <= vp + hw)
    ax.fill_between(g["v"][m], 0, g["absd"][m], color=C1, alpha=0.2, lw=0, zorder=1.5)
    ax.vlines([vp - hw, vp + hw], 0, np.interp([vp - hw, vp + hw], g["v"][::-1], g["absd"][::-1]),
              color=C1, lw=1, zorder=3)
    _arrow(ax, (vp - hw, 1.1 * h), (vp + hw, 1.1 * h), lw=1.1)
    _label(ax, vp, 1.1 * h, f"±{hw * 1000:.0f} mV", ha="center", size=7.5)


def draw_fwhm(ax, g):
    (v1, v2), y = g["half_v"], g["half_y"]
    _setup(ax, g, lo=v1 - 0.03)
    vp, top, ref = g["position"], g["neg"][g["idx"]], g["ref_level"]
    ax.vlines(vp, max(ref, ax.get_ylim()[0]), top, color=AXIS, lw=1, zorder=1.5)
    _arrow(ax, (v1, y), (v2, y))
    ax.plot([v1, v2], [y, y], "o", ms=4.5, color=C1, mec=SURFACE, mew=1, zorder=5)
    _label(ax, (v1 + v2) / 2, y - 0.06 * g["height"], f"{(v2 - v1) * 1000:.1f} mV",
           ha="center", va="top", size=7.5)


def draw_slope(ax, g, side):
    _setup(ax, g)
    vp, h = g["position"], g["height"]
    x2, y2 = (vp - FLANK, g["flank_y"][0]) if side == "left" else (vp + FLANK, g["flank_y"][1])
    ax.hlines(y2, min(x2, vp), max(x2, vp), color=MUTED, lw=0.9, linestyles=(0, (2, 2)), zorder=3)
    ax.vlines(vp, y2, h, color=MUTED, lw=0.9, linestyles=(0, (2, 2)), zorder=3)
    ax.plot([x2, vp], [y2, h], color=C1, lw=2, zorder=4)
    _dot(ax, vp, h)
    _dot(ax, x2, y2, ms=5.5)
    _label(ax, (x2 + vp) / 2, y2 - 0.05 * h, "25 mV", ha="center", va="top", color=INK2,
           size=7.5)
    ax.text(0.02, 0.97, f"drop {h - y2:.2f} Ah/V over 25 mV", transform=ax.transAxes,
            va="top", fontsize=7.5, color=INK2)


DRAWERS = {
    "height": draw_height, "position": draw_position, "prominence": draw_prominence,
    "area": lambda ax, g: draw_area(ax, g, "area"),
    "area35": lambda ax, g: draw_area(ax, g, "area35"),
    "area25": lambda ax, g: draw_area(ax, g, "area25"),
    "fwhm": draw_fwhm,
    "slope_left": lambda ax, g: draw_slope(ax, g, "left"),
    "slope_right": lambda ax, g: draw_slope(ax, g, "right"),
}


def _feature_title(ax, name, value):
    unit, fmt, desc = FEATURE_INFO[name]
    ax.set_title(f"{name} = {fmt.format(value)} {unit}", loc="left", fontsize=10,
                 fontweight="semibold", color=INK, pad=14)
    ax.text(0, 1.015, desc, transform=ax.transAxes, fontsize=7.8, color=INK2, va="bottom")


def draw_overview(ax, g):
    """All the construction at once (compact): ±50 mV window, apex, half-prominence width,
    flank secants."""
    _setup(ax, g, lo=g["half_v"][0] - 0.03)
    vp, h, hw = g["position"], g["height"], AREA_WINDOWS["area"]
    m = (g["v"] >= vp - hw) & (g["v"] <= vp + hw)
    ax.fill_between(g["v"][m], 0, g["absd"][m], color=C1, alpha=0.15, lw=0, zorder=1.5)
    ax.vlines(vp, 0, h, color=C1, lw=1.6, zorder=3)
    _arrow(ax, (g["half_v"][0], g["half_y"]), (g["half_v"][1], g["half_y"]), lw=1.2)
    ax.plot([vp - FLANK, vp, vp + FLANK], [g["flank_y"][0], h, g["flank_y"][1]], color=C1,
            lw=1.4, zorder=4)
    _dot(ax, vp, h)
    ax.set_xlabel("Voltage (V)")
    ax.set_ylabel("−dQ/dV (Ah/V)")
    ax.legend(handles=[
        Line2D([], [], color=C1, lw=1.6, marker="o", ms=5, mec=SURFACE, label="height, position"),
        Patch(color=C1, alpha=0.15, label="±50 mV window (area)"),
        Line2D([], [], color=C1, lw=1.2, label="fwhm (arrow)"),
        Line2D([], [], color=C1, lw=1.4, label="flank secants (slopes)"),
    ], loc="upper left", fontsize=7.5, handlelength=1.6)


def _values_panel(ax, g):
    ax.axis("off")
    rows = [f"{n:<12}{FEATURE_INFO[n][1].format(g[n]):>9}  {FEATURE_INFO[n][0]}"
            for n in GRID_ORDER]
    ax.text(0, 1, "extracted values", transform=ax.transAxes, va="top", fontsize=9.5,
            fontweight="semibold")
    ax.text(0, 0.9, "\n".join(rows), transform=ax.transAxes, va="top", family="monospace",
            fontsize=8.3, color=INK, linespacing=1.55)

### 3.3 The two views

- `show_cycle(cell, cycle)`: all nine features on one cycle. The bottom row puts the cycle among
  the cell's other cycles and shows its SoH label.
- `show_feature(cell, cycle, feature, x="cycle")`: one feature in three panels. The left panel is
  its geometry. The middle panel is its trajectory over the cell's life against every other cell
  in grey (`x="soh"` plots it against the SoH label). The right panel shows the history features
  at this cycle, drawn on the cycles they use.

A cycle with no curve or no peak (the 42 extraction failures) gets its §4 diagnostic figure
instead.

In [ ]:
REASONS = {  # reason: (short title, explanation)
    "bad_dqdv": (
        "empty placeholder cycle",
        "Cycle 0 of every batch-1 cell. Each of its nine channels (I, V, Qd, dQdV, ...) holds two "
        "zeros: the cycle was never recorded. The extractor needs a 1000-point dQ/dV, so the row "
        "is tagged <code>bad_dqdv</code>. Nothing can be extracted."),
    "no_peak": (
        "logging gap: flat curve, no peak",
        "b2c27 cycle 118. The logger stored only 18 discharge samples, all between 2.04 and "
        "2.00 V, so the capacity-vs-voltage curve and its dQ/dV are flat. No peak reaches the "
        "0.3 Ah/V prominence threshold (the largest is 0.0067), so the row is tagged "
        "<code>no_peak</code>."),
    "soh_nan": (
        "no SoH label (NaN)",
        "Cycle 0 of every batch-2 and batch-3 cell. Its features extract normally. The label "
        "lookup pairs cycle k with the summary row whose cycle number is k, but summary cycle "
        "numbers start at 1, so cycle 0 finds no row and its label is NaN. Its own capacity is "
        "summary row 0 (see 4.2)."),
    "soh_low": (
        "SoH label 0",
        "Cycle 1 of every batch-1 cell. Its features extract normally. The lookup gives it "
        "summary row 0, which is the empty cycle 0 above (QD = 0), so its label is 0. Its own "
        "capacity (row 1) gives SoH 1.000."),
    "soh_high": (
        "SoH label above 1.05 (capacity spike)",
        "Four cycles whose label is a capacity spike: summary row k−1 is 1.4 to 2.7 times the "
        "normal capacity because cycle k−1 was an abnormal recording, 9 to 45 hours long instead "
        "of about one. The one-cycle offset gives that spike to cycle k, which has a normal "
        "capacity and is dropped. Cycle k−1, the abnormal one, keeps a normal-looking label and "
        "stays in the training data."),
}
REASON_ORDER = list(REASONS)


def _new_figure(w, h, nrows, ncols, header=1.05, bottom=0.55, left=0.62, right=0.25, **gs_kw):
    """Figure + GridSpec with margins in inches, leaving `header` inches for the title."""
    fig = plt.figure(figsize=(w, h))
    gs = fig.add_gridspec(nrows, ncols, left=left / w, right=1 - right / w,
                          top=1 - header / h, bottom=bottom / h, **gs_kw)
    return fig, gs


def cycle_flags(cid, k):
    """The section-6 checks this cycle fails, as one short line ('' if none)."""
    s = SCAN.loc[(cid, k)]
    bits = []
    if s["block_points"] >= BLOCK_POINTS:
        bits.append(f"published dQ/dV {s['block_ratio']:.1f}× its Qdlin slope at "
                    f"{s['block_lo_v']:.3f}–{s['block_hi_v']:.3f} V")
    if s["hours"] > LONG_RECORDING_H:
        bits.append(f"{s['hours']:.1f} h recording")
    if s["discharge_pieces"] > 1:
        bits.append(f"discharge in {int(s['discharge_pieces'])} pieces")
    if s["qd_start"] > 0.05:
        bits.append(f"Qd starts at {s['qd_start']:.2f} Ah")
    return " · ".join(bits) + " (§6)" if bits else ""


def _header(fig, cid, k, g=None):
    r = FEAT.loc[(cid, k)]
    h = fig.get_figheight()
    label = "NaN" if pd.isna(r["SOH"]) else f"{r['SOH']:.4f}"
    fig.text(0.01, 1 - 0.14 / h, f"{cid}  ·  cycle {k}  ·  SoH label {label}", fontsize=13,
             fontweight="semibold", color=INK, va="top")
    meta = META.loc[cid]
    fig.text(0.01, 1 - 0.5 / h, f"batch {cid[1]}  ·  charge policy {meta['policy']}  ·  "
             f"{meta['n_cycles']:,} cycles in the data", fontsize=9, color=INK2, va="top")
    if r["reason"] == "kept":
        status, color = "kept for modelling", INK2
    else:
        status, color = f"✕ dropped by cleaning: {REASONS[r['reason']][0]} (see §4)", CRITICAL
    fig.text(0.99, 1 - 0.14 / h, status, fontsize=10, color=color, va="top", ha="right")
    warnings = [w for w in (cycle_flags(cid, k),) if w]
    if g is not None and not g["matches_table"]:
        warnings.insert(0, "recomputed values differ from the cached table")
    if warnings:
        fig.text(0.99, 1 - 0.5 / h, "⚠ " + " · ".join(warnings), fontsize=8.5, va="top",
                 ha="right", color=CRITICAL)
    elif g is not None:
        fig.text(0.99, 1 - 0.5 / h, "drawn values = cached feature table ✓", fontsize=8.5,
                 va="top", ha="right", color=MUTED)


def _draw_ageing(ax, g):
    """Whole ROI: the cycle in black over ~14 cycles spread across the cell's life."""
    curves = CURVES[g["cid"]]
    have = np.flatnonzero(~np.isnan(curves[:, 0]))
    pick = np.unique(have[np.linspace(0, have.size - 1, 14).round().astype(int)])
    norm = Normalize(have[0], have[-1])
    top, body = g["height"], V_ROI >= EDGE_V
    for j in pick:
        y = -curves[j][ROI_MASK]
        ax.plot(V_ROI, y, color=BLUES(norm(j)), lw=0.9, zorder=1)
        top = max(top, float(y[body].max()))
    ax.axhline(0, color=AXIS, lw=0.8, zorder=1)
    ax.plot(V_ROI, g["neg"], color=INK, lw=1.7, zorder=3)
    vp = g["position"]
    lo, hi = max(vp - 0.16, V_ROI[-1]), min(vp + 0.13, V_ROI[0])
    ax.add_patch(Rectangle((lo, -0.02 * top), hi - lo, 1.04 * g["height"], fill=False,
                           ec=MUTED, lw=0.9, ls=(0, (3, 2)), zorder=4))
    ax.set_xlim(V_ROI[-1], V_ROI[0])
    ax.set_ylim(min(0.0, float(g["neg"].min())) - 0.03 * top, 1.12 * top)
    ax.set_xlabel("Voltage (V)")
    ax.set_ylabel("−dQ/dV (Ah/V)")
    ax.set_title(f"whole ROI: this cycle (black) among {pick.size} cycles across the cell's life;"
                 " dashed box = zoom above", loc="left", color=INK)
    cax = ax.inset_axes([0.03, 0.86, 0.24, 0.04])
    cb = ax.figure.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=BLUES), cax=cax,
                            orientation="horizontal")
    cb.set_label("cycle of the blue lines", fontsize=7.5, color=INK2, labelpad=2)
    cb.ax.tick_params(labelsize=7, length=2)
    cb.outline.set_visible(False)


def _draw_soh(ax, cid, k):
    rows = FEAT.loc[cid]
    kept = rows[rows["reason"] == "kept"]
    ax.plot(kept.index, kept["SOH"], color=INK2, lw=1.1, zorder=2)
    if rows.loc[k, "reason"] == "kept":
        ax.axvline(k, color=MUTED, lw=0.8, zorder=1)
        _dot(ax, k, rows.loc[k, "SOH"], color=C1)
    else:
        ax.axvline(k, color=CRITICAL, lw=1, zorder=1)
    dropped = rows.index[rows["reason"] != "kept"]
    ax.text(0.98, 0.97, "dropped in this cell: cycle " + ", ".join(map(str, dropped)),
            transform=ax.transAxes, ha="right", va="top", fontsize=7.5, color=INK2)
    ax.set_xlabel("cycle")
    ax.set_ylabel("SoH label")
    ax.set_title("SoH label of the kept rows; line = this cycle", loc="left", color=INK)


def show_cycle(cid, k):
    """All nine features on one cycle (a §4 diagnostic for the 42 extraction failures)."""
    k = int(k)
    if FEAT.loc[(cid, k), "reason"] in ("bad_dqdv", "no_peak"):
        return show_dropped(cid, k)
    g = geometry(cid, k)
    fig, gs = _new_figure(13.5, 12.9, 4, 3, header=1.3, bottom=0.55, hspace=0.5, wspace=0.24,
                          height_ratios=[1, 1, 1, 1.05])
    for j, name in enumerate(GRID_ORDER):
        ax = fig.add_subplot(gs[j // 3, j % 3])
        DRAWERS[name](ax, g)
        _feature_title(ax, name, g[name])
        if j % 3 == 0:
            ax.set_ylabel("−dQ/dV (Ah/V)")
        if j >= 6 or name == "prominence":
            ax.set_xlabel("Voltage (V)")
    _draw_ageing(fig.add_subplot(gs[3, :2]), g)
    _draw_soh(fig.add_subplot(gs[3, 2]), cid, k)
    _header(fig, cid, k, g)
    return fig


# Grey background of every cell's kept rows, one feature at a time (cached).
_BACKGROUND = {}


def _background(name, x):
    if (name, x) not in _BACKGROUND:
        xs, ys = [], []
        for _, rows in clean.groupby("cell_id", sort=False):
            xs += [rows["cycle" if x == "cycle" else "SOH"].to_numpy(float), [np.nan]]
            ys += [rows[name].to_numpy(float), [np.nan]]
        _BACKGROUND[(name, x)] = (np.concatenate(xs), np.concatenate(ys))
    return _BACKGROUND[(name, x)]


def _fmt(v):
    return f"{v:+.4g}"


def _draw_trajectory(ax, cid, k, name, x, hist_k):
    bx, by = _background(name, x)
    if x == "cycle":
        ax.plot(bx, by, color=AXIS, lw=0.5, alpha=0.55, zorder=1)
    else:
        ax.plot(bx, by, ".", ms=1.2, color=AXIS, alpha=0.35, zorder=1, rasterized=True)
    rows = FEAT.loc[cid]
    kept = rows[rows["reason"] == "kept"]
    xs = kept.index.to_numpy(float) if x == "cycle" else kept["SOH"].to_numpy()
    ax.plot(xs, kept[name], color=INK2, lw=1.1, zorder=2)
    background = (dict(color=AXIS, lw=1.5) if x == "cycle"
                  else dict(color=AXIS, ls="none", marker=".", ms=6))
    handles = [Line2D([], [], **background, label=f"all {len(CELL_IDS)} cells (kept rows)"),
               Line2D([], [], color=INK2, lw=1.5, label=f"{cid} (kept rows)")]
    lost = rows[(rows["reason"] != "kept") & rows[name].notna()]
    if x == "cycle" and len(lost):
        ax.plot(lost.index, lost[name], "x", ms=7, mew=1.8, color=CRITICAL, zorder=4)
        handles.append(Line2D([], [], ls="none", marker="x", ms=7, mew=1.8, color=CRITICAL,
                              label="dropped rows of this cell"))
    fk = rows.loc[k, name]
    xk = k if x == "cycle" else rows.loc[k, "SOH"]
    if np.isfinite(xk) and rows.loc[k, "reason"] == "kept":
        _dot(ax, xk, fk, color=INK, ms=7)
        handles.append(Line2D([], [], ls="none", marker="o", ms=7, color=INK, mec=SURFACE,
                              label=f"cycle {k}"))
    if x == "cycle" and hist_k.get("status") == "ok":
        first = hist_k["first"]
        f0 = hist_k["grid"].loc[first]
        ax.hlines(f0, first, k, color=INK, lw=0.8, linestyles=(0, (1.5, 2)), zorder=3)
        _arrow(ax, (k, f0), (k, fk), color=INK, style="->", lw=1.4)
        handles.append(Line2D([], [], color=INK, lw=1.4, marker=r"$\uparrow$", ms=8,
                              label=f"drift = {_fmt(hist_k['drift'])}"))
    lo, hi = np.nanpercentile(by, [1, 99])
    lo, hi = min(lo, kept[name].min()), max(hi, kept[name].max())
    ax.set_ylim(lo - 0.04 * (hi - lo), hi + 0.04 * (hi - lo))
    ax.set_xlabel("cycle" if x == "cycle" else "SoH label")
    ax.set_ylabel(f"{name} ({FEATURE_INFO[name][0]})")
    ax.set_title(f"{name} over the cell's life", loc="left", fontsize=10,
                 fontweight="semibold", pad=14)
    ax.text(0, 1.015, f"this cell in dark grey, all {len(CELL_IDS)} cells in light grey", fontsize=7.8,
            color=INK2, transform=ax.transAxes, va="bottom")
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=2,
              fontsize=7.8, handlelength=1.6, columnspacing=1.2)


def _draw_history(ax, cid, k, name, h):
    ax.set_title("history features at this cycle", loc="left", fontsize=10,
                 fontweight="semibold", pad=14)
    ax.text(0, 1.015, "10-cycle window orange, 20-cycle window green", fontsize=7.8,
            color=INK2, transform=ax.transAxes, va="bottom")
    if h["status"] == "dropped":
        ax.text(0.5, 0.5, "row dropped by cleaning:\nno history features", ha="center",
                va="center", transform=ax.transAxes, color=CRITICAL, fontsize=9.5)
        ax.set_xticks([]), ax.set_yticks([])
        return
    grid, first = h["grid"], h["first"]
    lo = max(first, k - 23)
    win = grid.loc[lo:k]
    filled = h["filled"][(grid.index >= lo) & (grid.index <= k)]
    ax.plot(win.index, win.values, color=INK2, lw=0.9, zorder=2)
    ax.plot(win.index[~filled], win.values[~filled], "o", ms=3.5, color=INK2, zorder=3)
    if filled.any():
        ax.plot(win.index[filled], win.values[filled], "o", ms=3.5, mfc=SURFACE, mec=INK2,
                zorder=3)
    ax.set_xlabel("cycle")
    ax.set_ylabel(f"{name} ({FEATURE_INFO[name][0]})")
    if h["status"] == "short":
        ax.text(0.5, 0.95, f"only {k - first} cycles of clean history:\nhistory features start "
                f"at cycle {first + 20}", ha="center", va="top", transform=ax.transAxes,
                color=INK2, fontsize=8.5, zorder=6,
                bbox=dict(boxstyle="round,pad=0.3", fc=SURFACE, ec=AXIS, lw=0.6))
        ax.set_xlim(lo - 0.5, k + 3)
        return
    fk = h["f_k"]
    ax.hlines(h["rm20"], k - 19, k, color=C3, lw=5, alpha=0.3, zorder=1)
    xs = np.array([k - 19.0, k])
    ax.plot(xs, h["rm20"] + h["slope20"] * (xs - (k - 9.5)), color=C3, lw=1.6, zorder=4)
    m10 = float(grid.loc[k - 9:k].mean())
    xs = np.array([k - 9.0, k])
    ax.plot(xs, m10 + h["slope10"] * (xs - (k - 4.5)), color=C2, lw=1.6, zorder=4)
    for n_lag, color, dx in ((10, C2, 0.9), (20, C3, 1.9)):
        f_lag = float(grid.loc[k - n_lag])
        ax.hlines(f_lag, k - n_lag, k + dx, color=color, lw=0.8, linestyles=(0, (1.5, 2)),
                  zorder=3)
        _arrow(ax, (k + dx, f_lag), (k + dx, fk), color=color, style="->", lw=1.4)
    _dot(ax, k, fk, color=INK, ms=7)
    ax.set_xlim(lo - 0.5, k + 3)
    unit = FEATURE_INFO[name][0]
    handles = [
        Line2D([], [], ls="none", marker="o", ms=7, color=INK, mec=SURFACE,
               label=f"{name}(k) = {fk:.4g} {unit}"),
        Line2D([], [], color=C2, lw=1.4, marker=r"$\uparrow$", ms=8,
               label=f"d10 = {_fmt(h['d10'])}"),
        Line2D([], [], color=C3, lw=1.4, marker=r"$\uparrow$", ms=8,
               label=f"d20 = {_fmt(h['d20'])}"),
        Line2D([], [], color=C2, lw=1.6, label=f"slope10 = {_fmt(h['slope10'])} /cycle"),
        Line2D([], [], color=C3, lw=1.6, label=f"slope20 = {_fmt(h['slope20'])} /cycle"),
        Patch(color=C3, alpha=0.3, label=f"rm20 = {h['rm20']:.4g}"),
    ]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=2,
              fontsize=7.8, handlelength=1.6, columnspacing=1.2)


def show_feature(cid, k, name="height", x="cycle"):
    """One feature: its geometry, its trajectory over the cell's life, and its history."""
    k = int(k)
    if FEAT.loc[(cid, k), "reason"] in ("bad_dqdv", "no_peak"):
        return show_dropped(cid, k)
    g = geometry(cid, k)
    h = history_at(cid, k, name)
    fig, gs = _new_figure(14.5, 5.9, 1, 3, header=1.3, bottom=1.45, wspace=0.27,
                          width_ratios=[1, 1.2, 1])
    ax = fig.add_subplot(gs[0])
    DRAWERS[name](ax, g)
    _feature_title(ax, name, g[name])
    ax.set_xlabel("Voltage (V)")
    ax.set_ylabel("−dQ/dV (Ah/V)")
    _draw_trajectory(fig.add_subplot(gs[1]), cid, k, name, x, h)
    _draw_history(fig.add_subplot(gs[2]), cid, k, name, h)
    _header(fig, cid, k, g)
    return fig

### 3.4 Check: the drawings show the cached values

Recompute 500 random successful rows and compare them with `results/features_expanded.parquet`.
Also confirm that the drawn landmarks reproduce the values: the width between the half-prominence
crossings, apex minus reference level, and flank drop ÷ 25 mV.

In [ ]:
sample = feat[feat["status"] == "ok"].sample(500, random_state=0)
n_same, worst = 0, 0.0
for r in sample.itertuples():
    g = geometry(r.cell_id, int(r.cycle))
    n_same += g["matches_table"]
    worst = max(worst,
                abs((g["half_v"][1] - g["half_v"][0]) - g["fwhm"]),
                abs((g["neg"][g["idx"]] - g["ref_level"]) - g["prominence"]),
                abs((g["height"] - g["flank_y"][0]) / FLANK - g["slope_left"]) / 100,
                abs((g["height"] - g["flank_y"][1]) / FLANK - g["slope_right"]) / 100)
print(f"{n_same}/{len(sample)} sampled rows: all nine recomputed values identical to the "
      f"cached table")
print(f"largest landmark-vs-value mismatch: {worst:.1e}")
assert n_same == len(sample) and worst < 1e-9

### 3.5 Example

A static example (b1c1, cycle 100) for reading the figure. Call the function with any cell and
cycle; it returns the figure, e.g. `show_cycle("b3c5", 400).savefig("b3c5_c400.png")`.

In [ ]:
fig = show_cycle("b1c1", 100)

In [ ]:
fig = show_feature("b1c1", 100, "area25")

### 3.6 Interactive viewer

Pick a battery and a cycle. **View** switches between all nine features and one feature with its
trajectory and history. **Save PNG** writes the current view to `views/` next to this notebook.
The header flags in red the rows the cleaning step drops, and cycles that fail a §6 check; for
the 42 failures the viewer shows the §4 diagnostic instead.

In [ ]:
def _cell_options(batch):
    return [(f"{c}  ·  {META.loc[c, 'n_cycles']:,} cycles  ·  {META.loc[c, 'policy']}", c)
            for c in CELL_IDS if c.startswith(batch)]


_wide = {"description_width": "70px"}


class CellCycleControls:
    """Battery -> cell -> cycle pickers that stay consistent with each other; `on_change()` runs
    once after every change. The viewers of sections 3 and 5 both use it."""

    def __init__(self, cell, cycle, on_change):
        self.on_change, self._busy = on_change, False
        last = int(META.loc[cell, "n_cycles"]) - 1
        self.batch = widgets.ToggleButtons(
            options=[("Batch 1", "b1"), ("Batch 2", "b2"), ("Batch 3", "b3")], value=cell[:2],
            description="Battery", style=_wide)
        self.cell = widgets.Dropdown(options=_cell_options(cell[:2]), value=cell,
                                     description="Cell", style=_wide,
                                     layout=widgets.Layout(width="430px"))
        self.cycle = widgets.IntSlider(value=min(cycle, last), min=0, max=last,
                                       description="Cycle", style=_wide, continuous_update=False,
                                       layout=widgets.Layout(width="560px"))
        self.box = widgets.BoundedIntText(value=min(cycle, last), min=0, max=last,
                                          layout=widgets.Layout(width="90px"))
        widgets.link((self.cycle, "value"), (self.box, "value"))
        self.back = widgets.Button(description="◀", tooltip="previous cycle",
                                   layout=widgets.Layout(width="44px"))
        self.fwd = widgets.Button(description="▶", tooltip="next cycle",
                                  layout=widgets.Layout(width="44px"))
        self.batch.observe(self._on_batch, names="value")
        self.cell.observe(self._on_cell, names="value")
        self.cycle.observe(self._on_cycle, names="value")
        self.back.on_click(lambda _: self.step(-1))
        self.fwd.on_click(lambda _: self.step(+1))
        self.widget = widgets.VBox([widgets.HBox([self.batch, self.cell]),
                                    widgets.HBox([self.cycle, self.box, self.back, self.fwd])])

    @property
    def value(self):
        return self.cell.value, int(self.cycle.value)

    def step(self, delta):
        self.cycle.value = int(np.clip(self.cycle.value + delta, self.cycle.min, self.cycle.max))

    def _on_cycle(self, *_):
        if not self._busy:
            self.on_change()

    def _on_cell(self, *_):
        if self._busy:
            return
        self._busy = True
        try:
            last = int(META.loc[self.cell.value, "n_cycles"]) - 1
            self.cycle.value = min(int(self.cycle.value), last)
            self.cycle.max = last
            self.box.max = last
        finally:
            self._busy = False
        self.on_change()

    def _on_batch(self, *_):
        self._busy = True
        try:
            self.cell.options = _cell_options(self.batch.value)
            self.cell.value = self.cell.options[0][1]
        finally:
            self._busy = False
        self._on_cell()


def render_into(out, make_figure):
    """Replace what an Output widget shows with a fresh figure."""
    with out:
        out.clear_output(wait=True)
        fig = make_figure()
        display(fig)
        plt.close(fig)


def save_png(fig, name, message):
    """Write a figure to views/ next to this notebook and say where in `message`."""
    path = NB_DIR / "views" / f"{name}.png"
    path.parent.mkdir(exist_ok=True)
    fig.savefig(path)
    plt.close(fig)
    message.value = f"saved <code>{path.relative_to(REPO_ROOT).as_posix()}</code>"


view_w = widgets.ToggleButtons(options=[("All nine features", "nine"),
                                        ("One feature + history", "one")],
                               value="nine", description="View", style=_wide)
feat_w = widgets.Dropdown(options=GRID_ORDER, value="height", description="Feature",
                          style=_wide, layout=widgets.Layout(width="220px"))
xaxis_w = widgets.ToggleButtons(options=[("vs cycle", "cycle"), ("vs SoH", "soh")],
                                value="cycle", style={"button_width": "80px"})
save_w = widgets.Button(description="Save PNG", icon="download",
                        tooltip="save this view to the views/ folder next to the notebook")
msg_w = widgets.HTML()
out_w = widgets.Output()


def _main_figure():
    cid, k = main.value
    if view_w.value == "nine":
        return show_cycle(cid, k)
    return show_feature(cid, k, feat_w.value, xaxis_w.value)


def _render(*_):
    render_into(out_w, _main_figure)


def _on_view(*_):
    one = view_w.value == "one"
    feat_w.layout.display = None if one else "none"
    xaxis_w.layout.display = None if one else "none"
    _render()


def _save(*_):
    cid, k = main.value
    tag = "nine" if view_w.value == "nine" else f"{feat_w.value}_vs_{xaxis_w.value}"
    save_png(_main_figure(), f"{cid}_cycle{k}_{tag}", msg_w)


main = CellCycleControls("b1c1", 100, _render)
view_w.observe(_on_view, names="value")
feat_w.observe(_render, names="value")
xaxis_w.observe(_render, names="value")
save_w.on_click(_save)
display(widgets.VBox([main.widget, widgets.HBox([view_w, feat_w, xaxis_w]),
                      widgets.HBox([save_w, msg_w])]), out_w)
_on_view()

## 4. The 170 dropped rows

`clean_feature_table` applies four filters in order: status `ok`, then SoH not NaN, then
SoH > 0.5, then SoH ≤ 1.05. Each of the 170 rows it removes falls under one of five reasons. Only
the first two are extraction failures; the other three have normal features and lose the row
because of the SoH label.

In [ ]:
def _which(d):
    """'cycle 0 of every b1 cell' when that is what the rows are, else a list."""
    if d["cycle"].nunique() == 1:
        batches = sorted(d["batch"].unique())
        if all(d.loc[d["batch"] == b, "cell_id"].nunique() == sum(c.startswith(b) for c in CELL_IDS)
               for b in batches):
            return f"cycle {d['cycle'].iloc[0]} of every {' and '.join(batches)} cell"
    return ", ".join(f"{c} cycle {k}" for c, k in zip(d["cell_id"], d["cycle"]))


summary = pd.DataFrame([{
    "reason": r,
    "what it is": REASONS[r][0],
    "which rows": _which(d),
    "features extracted": "yes" if (d["status"] == "ok").all() else "no",
    **{b: int((d["batch"] == b).sum()) for b in ("b1", "b2", "b3")},
    "rows": len(d),
} for r in REASON_ORDER for d in [DROPPED[DROPPED["reason"] == r]]]).set_index("reason")
summary.loc["total"] = ["", "", ""] + summary[["b1", "b2", "b3", "rows"]].sum().tolist()
display(summary)

### 4.1 One example of each reason

The browser in §4.4 shows any of the 170 rows.

In [ ]:
def _draw_label_lookup(ax, cid, k, j_range):
    """Summary QD by row index, marking cycle k's own row (k) and the row the extractor reads
    for it (k-1)."""
    qd, j = QD[cid], np.arange(len(QD[cid]))
    sel = (j >= j_range[0]) & (j <= j_range[1])
    ax.plot(j[sel], qd[sel], color=INK2, lw=1, marker="o", ms=3.5, zorder=2)
    ax.plot([k], [qd[k]], "o", ms=11, mfc="none", mec=C1, mew=2, zorder=4,
            label=f"row {k}: cycle {k}'s own capacity (= max of its Qd trace)")
    if k >= 1:
        ax.plot([k - 1], [qd[k - 1]], "x", ms=10, mew=2.2, color=CRITICAL, zorder=5,
                label=f"row {k - 1}: the label the extractor gives cycle {k}")
    handles, _ = ax.get_legend_handles_labels()
    if k == 0:
        handles.append(Line2D([], [], ls="none", marker="x", ms=10, mew=2.2, color=CRITICAL,
                              label="no row −1: the extractor looks for summary cycle 0, but the\n"
                                    "numbering starts at 1, so cycle 0 gets no label (NaN)"))
    ax.set_xlabel("summary row j  (its cycle number is j + 1)")
    ax.set_ylabel("summary QD (Ah)")
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=7.8)


def _fig_placeholder(cid, k):
    fig, gs = _new_figure(14.5, 5.2, 1, 3, bottom=1.25, wspace=0.3, width_ratios=[0.85, 1.1, 1])
    ax0, ax1, ax2 = (fig.add_subplot(gs[i]) for i in range(3))
    ax0.axis("off")
    rows = [f"{ch:<6} {v.size} values  {np.array2string(v, precision=1)}"
            for ch, v in RAW[(cid, k)].items()]
    ax0.text(0, 1, f"what the pickle holds for cycle {k}", transform=ax0.transAxes, va="top",
             fontsize=10, fontweight="semibold")
    ax0.text(0, 0.88, "\n".join(rows), transform=ax0.transAxes, va="top", family="monospace",
             fontsize=8.3, linespacing=1.5)
    ax0.text(0, 0.02, "Every channel is two zeros: the cycle was not\nrecorded. The extractor "
             "needs a 1000-point\ndQ/dV, so the row is tagged bad_dqdv.",
             transform=ax0.transAxes, va="bottom", fontsize=8.5, color=INK2)
    nxt = k + 1
    ax1.plot(V_ROI, -CURVES[cid][nxt][ROI_MASK], color=INK2, lw=1.3)
    ax1.axhline(0, color=AXIS, lw=0.8)
    ax1.set_title(f"cycle {nxt}, the first recorded cycle (cycle {k} has no curve)", loc="left")
    ax1.set_xlabel("Voltage (V)")
    ax1.set_ylabel("−dQ/dV (Ah/V)")
    _draw_label_lookup(ax2, cid, nxt, (0, 6))
    ax2.set_title(f"its summary capacity (row {k} = 0 Ah) becomes cycle {nxt}'s label", loc="left")
    return fig


def _fig_no_peak(cid, k):
    fig, gs = _new_figure(14.5, 5.0, 1, 3, bottom=1.05, wspace=0.28)
    ax0, ax1, ax2 = (fig.add_subplot(gs[i]) for i in range(3))
    for kk, color, ls in ((k - 1, INK2, "-"), (k + 1, MUTED, (0, (4, 2))), (k, CRITICAL, "-")):
        ax0.plot(V_ROI, -CURVES[cid][kk][ROI_MASK], color=color, ls=ls,
                 lw=1.7 if kk == k else 1.1, label=f"cycle {kk}")
    ax0.axhline(0, color=AXIS, lw=0.8)
    ax0.set_title(f"cycle {k} against its neighbours", loc="left")
    ax0.set_xlabel("Voltage (V)")
    ax0.set_ylabel("−dQ/dV (Ah/V)")
    ax0.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)
    for kk, color in ((k - 1, MUTED), (k, CRITICAL)):
        raw = RAW[(cid, kk)]
        m = raw["I"] < I_DISCHARGE_MAX
        ax1.plot(raw["Qd"][m], raw["V"][m], color=color, lw=1.1, marker="o",
                 ms=3 if kk == k else 0, label=f"cycle {kk}: {m.sum()} discharge samples")
    ax1.set_title("raw discharge samples (I < −0.5 A)", loc="left")
    ax1.set_xlabel("Qd (Ah)")
    ax1.set_ylabel("V (V)")
    ax1.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18))
    neg = -CURVES[cid][k][ROI_MASK]
    cand, _ = find_peaks(neg, distance=DISTANCE)
    proms = peak_prominences(neg, cand)[0]
    ax2.plot(V_ROI, neg, color=CRITICAL, lw=1.2)
    ax2.plot(V_ROI[cand], neg[cand], "v", ms=5, color=INK2)
    ax2.set_title(f"cycle {k} rescaled: {cand.size} local maxima", loc="left")
    ax2.text(0.97, 0.95, f"largest prominence {proms.max():.4f} Ah/V\n"
             f"threshold {PROMINENCE} Ah/V, so no peak", transform=ax2.transAxes, va="top",
             ha="right", fontsize=8.5, color=INK)
    ax2.set_xlabel("Voltage (V)")
    ax2.set_ylabel("−dQ/dV (Ah/V)")
    return fig


def _fig_label(cid, k):
    g = geometry(cid, k)
    fig, gs = _new_figure(14.5, 5.2, 1, 3, bottom=1.25, wspace=0.3, width_ratios=[1.1, 0.6, 1])
    ax0, ax1, ax2 = (fig.add_subplot(gs[i]) for i in range(3))
    draw_overview(ax0, g)
    ax0.set_title("the features extract normally (status ok)", loc="left")
    _values_panel(ax1, g)
    _draw_label_lookup(ax2, cid, k, (0, 6))
    ax2.set_title("where the SoH label comes from", loc="left")
    return fig


def _fig_spike(cid, k):
    fig, gs = _new_figure(14.5, 5.2, 1, 3, bottom=1.25, wspace=0.3)
    ax0, ax1, ax2 = (fig.add_subplot(gs[i]) for i in range(3))
    fate = {kk: "kept" if FEAT.loc[(cid, kk), "reason"] == "kept" else "dropped"
            for kk in (k - 2, k - 1, k)}
    for kk, color, lw, ls, z in ((k - 2, MUTED, 1.1, (0, (4, 2)), 3), (k - 1, C2, 1.7, "-", 2),
                                 (k, INK2, 1.3, "-", 4)):
        ax0.plot(V_ROI, -CURVES[cid][kk][ROI_MASK], color=color, lw=lw, ls=ls, zorder=z,
                 label=f"cycle {kk} ({fate[kk]})")
    ax0.axhline(0, color=AXIS, lw=0.8)
    ax0.set_title(f"curves: cycle {k - 1} is the abnormal recording", loc="left")
    ax0.set_xlabel("Voltage (V)")
    ax0.set_ylabel("−dQ/dV (Ah/V)")
    ax0.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3)
    _draw_label_lookup(ax1, cid, k, (k - 8, k + 6))
    ax1.set_title(f"cycle {k}'s label is cycle {k - 1}'s capacity", loc="left")
    for kk, color in ((k - 2, MUTED), (k - 1, C2)):
        raw = RAW[(cid, kk)]
        ax2.plot(raw["t"] / 60, raw["Qd"], color=color, lw=1.2,
                 label=f"cycle {kk}: {raw['t'].max() / 60:.1f} h, max Qd {raw['Qd'].max():.3f} Ah")
    ax2.set_title("raw discharge-capacity trace", loc="left")
    ax2.set_xlabel("time in the cycle (h)")
    ax2.set_ylabel("Qd (Ah)")
    ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2))
    return fig


_FIGS = {"bad_dqdv": _fig_placeholder, "no_peak": _fig_no_peak, "soh_nan": _fig_label,
         "soh_low": _fig_label, "soh_high": _fig_spike}


def show_dropped(cid, k):
    """Diagnostic figure for one of the 170 dropped rows."""
    k = int(k)
    reason = FEAT.loc[(cid, k), "reason"]
    if reason == "kept":
        raise ValueError(f"{cid} cycle {k} is kept; use show_cycle")
    fig = _FIGS[reason](cid, k)
    _header(fig, cid, k)
    return fig


for r in REASON_ORDER:
    first = DROPPED[DROPPED["reason"] == r].iloc[0]
    display(HTML(f"<b>{r}</b>: {REASONS[r][1]}"))
    fig = show_dropped(first["cell_id"], first["cycle"])
    plt.show()

### 4.2 The one-cycle offset in the SoH label

In the pickles, `cell["cycles"][str(k)]` is entry k of the MATLAB `cycles` struct, and summary row
k is entry k of the summary, whose `cycle` field reads k + 1 because MATLAB counts from 1. Both
are the same physical cycle: the code below checks that every cycle's own discharge capacity (the
maximum of its raw `Qd` trace) equals summary `QD` row k.

The extractors (`features.extract_all_features` and `peak_extraction.extract_features`) take
the label from the summary row whose `cycle` field equals k, which is row k − 1. **Every SoH label
in `features_expanded.parquet` is therefore the previous cycle's capacity.** This explains 124 of
the 128 label drops (cycle 0 has no row k − 1; batch-1 cycle 1 gets the empty cycle 0's 0 Ah). It
also explains why each capacity spike lands on the cycle after the abnormal one.

On kept rows the two labels differ by about 1e-4 SoH, which is small next to a test RMSE of about
0.016. The exceptions are the single-cycle spikes and dips, which move to the neighbouring row.
This notebook does not change the pipeline: every number in the thesis was computed with the
lagged labels.

In [ ]:
n = same = prev = 0
summary_ok = True
for cid in CELL_IDS:
    m, qd, ok = MAXQD[cid], QD[cid], np.isfinite(MAXQD[cid])
    summary_ok &= bool(np.array_equal(SUMMARY_CYCLE[cid], np.arange(1, len(qd) + 1)))
    n += int(ok.sum())
    same += int((m[ok] == qd[ok]).sum())
    prev += int((m[1:][ok[1:]] == qd[:-1][ok[1:]]).sum())
print(f"summary 'cycle' field equals row index + 1 in every cell: {summary_ok}")
print(f"cycles with a recorded Qd trace: {n:,}")
print(f"  max(Qd trace of cycle k) == summary QD row k     : {same:,}")
print(f"  max(Qd trace of cycle k) == summary QD row k - 1 : {prev:,}  <- the row the label uses")

kept = feat[feat["reason"] == "kept"]
diff = (kept["SOH"] - kept["SOH_own"]).abs()
print(f"\nlabel used vs the cycle's own capacity, over the {len(kept):,} kept rows:")
print(f"  median |difference| {diff.median():.1e}, 99th percentile {diff.quantile(0.99):.1e}, "
      f"max {diff.max():.3f}; {int((diff > 0.01).sum())} rows differ by more than 0.01")
own_clean, _ = clean_feature_table(feat.assign(SOH=feat["SOH_own"]), verbose=False)
print(f"rows the cleaning would drop with each cycle's own capacity: {len(feat) - len(own_clean)} "
      f"(42 extraction failures + the capacity spikes), instead of {len(feat) - len(clean)}")

### 4.3 The list

All 170 rows, with the label the pipeline used (`SOH`) and the cycle's own capacity as SoH
(`SOH_own`). Also written to `dropped_rows.csv` next to this notebook.

In [ ]:
dropped_table = DROPPED[["cell_id", "batch", "cycle", "reason", "status", "SOH", "SOH_own"]].copy()
dropped_table["dqdv_points"] = [int((~np.isnan(CURVES[c][k])).sum())
                                for c, k in zip(DROPPED["cell_id"], DROPPED["cycle"])]
dropped_table = dropped_table.sort_values(
    ["reason", "cell_id", "cycle"],
    key=lambda s: s.map({r: i for i, r in enumerate(REASON_ORDER)}) if s.name == "reason"
    else s.map(lambda c: (int(c[1]), int(c[3:]))) if s.name == "cell_id" else s,
).reset_index(drop=True)
dropped_table.to_csv(NB_DIR / "dropped_rows.csv", index=False)
print(f"wrote {(NB_DIR / 'dropped_rows.csv').relative_to(REPO_ROOT).as_posix()} "
      f"({len(dropped_table)} rows)")
with pd.option_context("display.max_rows", 200):
    display(dropped_table)

### 4.4 Browser for all 170

Choose a reason, then a row; ◀ / ▶ step through the rows of that reason.

In [ ]:
def _row_options(reason):
    d = dropped_table[dropped_table["reason"] == reason]
    return [(f"{c}  ·  cycle {k}", (c, int(k))) for c, k in zip(d["cell_id"], d["cycle"])]


reason_w = widgets.Dropdown(
    options=[(f"{REASONS[r][0]}  ({int((DROPPED['reason'] == r).sum())})", r)
             for r in REASON_ORDER],
    value="bad_dqdv", description="Reason", style=_wide, layout=widgets.Layout(width="420px"))
row_w = widgets.Dropdown(options=_row_options("bad_dqdv"), description="Row", style=_wide,
                         layout=widgets.Layout(width="260px"))
back2_w = widgets.Button(description="◀", layout=widgets.Layout(width="44px"))
fwd2_w = widgets.Button(description="▶", layout=widgets.Layout(width="44px"))
why_w = widgets.HTML(layout=widgets.Layout(width="960px"))
out2_w = widgets.Output()
_busy2 = {"on": False}


def _render_dropped(*_):
    if _busy2["on"]:
        return
    why_w.value = f"<div style='line-height:1.4'>{REASONS[reason_w.value][1]}</div>"
    render_into(out2_w, lambda: show_dropped(*row_w.value))


def _on_reason(*_):
    _busy2["on"] = True
    try:
        row_w.options = _row_options(reason_w.value)
        row_w.value = row_w.options[0][1]
    finally:
        _busy2["on"] = False
    _render_dropped()


def _step_row(delta):
    values = [v for _, v in row_w.options]
    i = values.index(row_w.value)
    row_w.value = values[int(np.clip(i + delta, 0, len(values) - 1))]


reason_w.observe(_on_reason, names="value")
row_w.observe(_render_dropped, names="value")
back2_w.on_click(lambda _: _step_row(-1))
fwd2_w.on_click(lambda _: _step_row(+1))
display(widgets.VBox([widgets.HBox([reason_w, row_w, back2_w, fwd2_w]), why_w]), out2_w)
_render_dropped()

## 5. Published vs recomputed dQ/dV

**Two curves.** The features are measured on the *published* dQ/dV: the `dQdV` array the Severson
dataset ships with every cycle (the authors' `discharge_dQdV`), on the 1000-point grid. How the
authors computed it is not documented. The *recomputed* curve is this project's own rebuild from
the raw time series, `recompute_dqdv` in `src/dqdv.py` (ENG-02):

1. keep the discharge samples (I < −0.5 A);
2. sort them by voltage and average any repeated voltage, so that Qd(V) is a function;
3. interpolate Qd onto the same 1000-point grid;
4. differentiate (`np.gradient`), which gives dQ/dV with the published sign;
5. smooth with a Savitzky–Golay filter (51 points ≈ 77 mV, order 3).

**Where it lives.** Nowhere on disk. `stages/s03_dqdv_check.py` used it to check the published
curve (median Pearson r 0.997; Fig. B1 and Table B1, `report/23_fig_B1_recomputed_vs_published.pdf`). ENG-05 kept only the comparison in
`features_expanded.parquet`: `rc_pearson_r` and `rc_rms_diff` (agreement with the published curve)
and a second-peak search on it (`rc_prom2`, `rc_position2`, `rc_n_peaks2`). No model feature comes
from the recomputed curve. This section rebuilds it on demand from the raw discharge samples kept
in §1.

**Reading it.** When the unsmoothed rebuild is as rough as the published curve, the noise is in the
measurements; b2c40 is like this. When the published curve has a step or block that the rebuild
does not have, the published curve is wrong there; b2c47 cycle 630 is like this (§6). Smoothing
removes noise but also rounds a sharp peak, so features measured on the smoothed curve differ from
the published ones even on a clean cell. At b2c1 cycle 100 the flank slopes move by 40–50 %, while height,
position, prominence and the areas move by a few percent at most. The slider shows how this
depends on the window.

### 5.1 Check: this is the curve ENG-05 compared

Rebuild 300 random cycles and recompute the two agreement statistics ENG-05 stored for them.

In [ ]:
def recompute(cid, k, window=SMOOTH_WINDOW):
    """src/dqdv.py's dQ/dV rebuilt from one cycle's raw discharge samples, on the published grid
    (window=None: not smoothed). All-NaN where recompute_dqdv returns all-NaN."""
    samples = DISCHARGE[cid][k]
    if samples is None:
        return np.full(V_GRID_POINTS, np.nan)
    return dqdv_from_raw(*samples, v_grid=V_GRID, smooth_window=window,
                         polyorder=SMOOTH_POLYORDER)


sample = feat[feat["rc_pearson_r"].notna()].sample(300, random_state=1)
worst = 0.0
for r in sample.itertuples():
    agree = compare_with_published(recompute(r.cell_id, r.cycle), CURVES[r.cell_id][r.cycle])
    worst = max(worst, abs(agree["pearson_r"] - r.rc_pearson_r),
                abs(agree["rms_diff"] - r.rc_rms_diff))
print(f"300 cycles: largest difference from ENG-05's rc_pearson_r and rc_rms_diff: {worst:.1e}")
assert worst < 1e-12

### 5.2 The figure

`show_recomputed(cell, cycle, window=51, feature="height")` has four panels:
- **peak region**: the published curve (grey), the unsmoothed rebuild (thin) and the smoothed
  rebuild (blue), with the peak the extractor tracks on each;
- **whole ROI**: the published curve and the smoothed rebuild, with their agreement;
- **features**: the nine values measured on the published curve and on the smoothed rebuild;
- **over the cell's life**: one feature measured on both curves for every cycle of the cell, with
  the cycles §6 flags as inconsistent marked. The rebuild goes through the extractor's own
  detection and cycle-to-cycle tracking (`features._dominant_peak_features`), so the two lines
  differ only in the curve they are measured on. The first view of a cell takes a second or two.

The example below is b2c47 cycle 630, where the published curve has a block near the peak and the
extractor measures the block.

In [ ]:
_RC_FEATURES = {}


def recomputed_features(cid, window=SMOOTH_WINDOW):
    """The nine features measured on the smoothed rebuild of every cycle of a cell, with the
    extractor's detection and tracking (as extract_all_features runs it). Cached per window."""
    if (cid, window) not in _RC_FEATURES:
        prev, rows = None, {}
        for k in range(len(CURVES[cid])):
            if DISCHARGE[cid][k] is None:
                continue
            vals = _dominant_peak_features(recompute(cid, k, window)[ROI_MASK], prev)
            if vals is not None:
                prev = vals["position"]
                rows[k] = vals
        _RC_FEATURES[(cid, window)] = pd.DataFrame.from_dict(rows, orient="index")[FEATURE_COLS]
    return _RC_FEATURES[(cid, window)]


def _finite_max(*arrays):
    tops = [float(np.nanmax(a)) for a in arrays if np.isfinite(a).any()]
    return max(tops) if tops else 1.0


def _feature_table(ax, published, recomputed):
    ax.axis("off")
    lines = [f"{'':<12}{'published':>10}{'recomputed':>11}{'change':>10}"]
    for f in GRID_ORDER:
        a, b = published.get(f, np.nan), recomputed.get(f, np.nan)
        fmt = FEATURE_INFO[f][1]
        shown_a = fmt.format(a) if np.isfinite(a) else "–"
        shown_b = fmt.format(b) if np.isfinite(b) else "–"
        if not (np.isfinite(a) and np.isfinite(b)):
            change = "–"
        elif f == "position":
            change = f"{(b - a) * 1000:+.1f} mV"
        else:
            rel = (b - a) / abs(a)
            change = f"{rel:+.1%}" if abs(rel) < 0.1 else f"{rel:+.0%}"
        lines.append(f"{f:<12}{shown_a:>10}{shown_b:>11}{change:>10}")
    ax.text(0, 1, "features at this cycle", transform=ax.transAxes, va="top", fontsize=10,
            fontweight="semibold")
    ax.text(0, 0.9, "\n".join(lines), transform=ax.transAxes, va="top", family="monospace",
            fontsize=8.1, linespacing=1.6)
    ax.text(0, 0.02, "recomputed = the smoothed rebuild, measured\nwith the same extractor and "
            "tracking", transform=ax.transAxes, va="bottom", fontsize=7.8, color=INK2)


def show_recomputed(cid, k, window=SMOOTH_WINDOW, name="height"):
    """The published dQ/dV against the one rebuilt from the raw samples at one cycle, and one
    feature measured on both curves over the cell's life."""
    k = int(k)
    rc_curve = recompute(cid, k, window)
    pub, raw, smooth = -CURVES[cid][k], -recompute(cid, k, None), -rc_curve
    rc = recomputed_features(cid, window)
    row = FEAT.loc[(cid, k)]
    published = {f: row[f] for f in FEATURE_COLS} if row["status"] == "ok" else {}
    recomputed = rc.loc[k].to_dict() if k in rc.index else {}
    unit = FEATURE_INFO[name][0]
    fig, gs = _new_figure(14.5, 10.0, 2, 3, bottom=1.2, hspace=0.36, wspace=0.28,
                          width_ratios=[1.1, 1.1, 0.78], height_ratios=[1, 0.85])
    ax_peak, ax_roi, ax_tab = (fig.add_subplot(gs[0, i]) for i in range(3))
    ax_life = fig.add_subplot(gs[1, :])

    centre = published.get("position", recomputed.get("position", 3.15))
    lo, hi = max(centre - 0.16, V_ROI[-1]), min(centre + 0.13, V_ROI[0])
    view = (V_GRID >= lo) & (V_GRID <= hi)
    ax_peak.axhline(0, color=AXIS, lw=0.8)
    ax_peak.plot(V_GRID, raw, color=MUTED, lw=0.7, alpha=0.75, label="recomputed, not smoothed")
    ax_peak.plot(V_GRID, pub, color=INK2, lw=1.3, label="published (the features use this)")
    ax_peak.plot(V_GRID, smooth, color=C1, lw=2, label=f"recomputed, smoothed ({window} points)")
    if published:
        _dot(ax_peak, published["position"], published["height"], color=INK2)
    if recomputed:
        _dot(ax_peak, recomputed["position"], recomputed["height"], color=C1)
    top = _finite_max(pub[view], smooth[view])
    ax_peak.set_xlim(lo, hi)
    ax_peak.set_ylim(-0.04 * top, 1.18 * top)
    ax_peak.set_title(f"peak region · Savitzky–Golay {window} points "
                      f"({window * DV * 1000:.0f} mV), order {SMOOTH_POLYORDER}", loc="left")
    ax_peak.set_xlabel("Voltage (V)")
    ax_peak.set_ylabel("−dQ/dV (Ah/V)")
    ax_peak.legend(loc="upper left", fontsize=7.6)

    body = ROI_MASK & (V_GRID >= EDGE_V)
    top = _finite_max(pub[body], smooth[body])
    ax_roi.axhline(0, color=AXIS, lw=0.8)
    ax_roi.plot(V_GRID, pub, color=INK2, lw=1.1)
    ax_roi.plot(V_GRID, smooth, color=C1, lw=1.6)
    ax_roi.set_xlim(V_ROI[-1], V_ROI[0])
    ax_roi.set_ylim(-0.04 * top, 1.12 * top)
    agree = compare_with_published(rc_curve, CURVES[cid][k])
    ax_roi.set_title("whole ROI" + (f" · r = {agree['pearson_r']:.3f}, RMS difference "
                                    f"{agree['rms_diff']:.2f} Ah/V"
                                    if np.isfinite(agree["pearson_r"]) else ""), loc="left")
    ax_roi.set_xlabel("Voltage (V)")
    if not (np.isfinite(pub).any() or np.isfinite(raw).any()):
        for ax in (ax_peak, ax_roi):
            ax.text(0.5, 0.5, "no recorded data for this cycle", transform=ax.transAxes,
                    ha="center", va="center", color=CRITICAL, fontsize=9.5)

    _feature_table(ax_tab, published, recomputed)

    rows = FEAT.loc[cid]
    rows = rows[rows["status"] == "ok"]
    ax_life.plot(rows.index, rows[name], color=INK2, lw=0.9, label="measured on the published curve")
    ax_life.plot(rc.index, rc[name], color=C1, lw=1.2,
                 label=f"measured on the recomputed curve ({window} points)")
    checks = SCAN.loc[cid]
    bad = [c for c in checks.index[checks["block_points"] >= BLOCK_POINTS] if c in rows.index]
    if bad:
        ax_life.plot(bad, rows.loc[bad, name], "x", ms=6, mew=1.6, color=CRITICAL, zorder=4,
                     label="published curve inconsistent (§6)")
    ax_life.axvline(k, color=MUTED, lw=0.8, zorder=1)
    if k in rows.index:
        _dot(ax_life, k, rows.loc[k, name], color=INK2)
    if k in rc.index:
        _dot(ax_life, k, rc.loc[k, name], color=C1)

    def step(values):
        return float(np.nanmedian(np.abs(np.diff(values.to_numpy(dtype=float)))))

    ax_life.set_title(f"{name} over the cell's life, measured on both curves", loc="left",
                      fontsize=10, fontweight="semibold", pad=14)
    ax_life.text(0, 1.015, f"typical change from one cycle to the next: published "
                 f"{step(rows[name]):.3g} {unit}, recomputed {step(rc[name]):.3g} {unit}",
                 transform=ax_life.transAxes, fontsize=7.8, color=INK2, va="bottom")
    ax_life.set_xlabel("cycle")
    ax_life.set_ylabel(f"{name} ({unit})")
    ax_life.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=3)
    _header(fig, cid, k)
    return fig


fig = show_recomputed("b2c47", 630)

### 5.3 Viewer

It starts on b2c40, the noisiest cell. **Smoothing** sets the Savitzky–Golay window (51 is
ENG-02's choice); **Feature** picks the feature for the lower panel.

In [ ]:
rc_window_w = widgets.IntSlider(value=SMOOTH_WINDOW, min=5, max=101, step=2,
                                description="Smoothing", style=_wide, continuous_update=False,
                                layout=widgets.Layout(width="420px"))
rc_feat_w = widgets.Dropdown(options=GRID_ORDER, value="height", description="Feature",
                             style=_wide, layout=widgets.Layout(width="220px"))
rc_save_w = widgets.Button(description="Save PNG", icon="download",
                           tooltip="save this view to the views/ folder next to the notebook")
rc_msg_w = widgets.HTML()
rc_out_w = widgets.Output()


def _rc_figure():
    cid, k = rc_ctl.value
    return show_recomputed(cid, k, rc_window_w.value, rc_feat_w.value)


def _render_rc(*_):
    render_into(rc_out_w, _rc_figure)


def _save_rc(*_):
    cid, k = rc_ctl.value
    save_png(_rc_figure(), f"{cid}_cycle{k}_recomputed_w{rc_window_w.value}_{rc_feat_w.value}",
             rc_msg_w)


rc_ctl = CellCycleControls("b2c40", 200, _render_rc)
rc_window_w.observe(_render_rc, names="value")
rc_feat_w.observe(_render_rc, names="value")
rc_save_w.on_click(_save_rc)
display(widgets.VBox([rc_ctl.widget, widgets.HBox([rc_window_w, rc_feat_w]),
                      widgets.HBox([rc_save_w, rc_msg_w])]), rc_out_w)
_render_rc()

## 6. Odd cycles

§1 ran these checks on every cycle. The first three judge single cycles; the last ranks whole
cells.

| check | rule | what it catches |
|---|---|---|
| published curve inconsistent | for at least 5 grid points in a row (7.5 mV) the published dQ/dV is more than 1.5 × the slope of the dataset's own capacity curve `Qdlin` (where that slope is above 0.5 Ah/V); normally the two agree within about 10 % | steps, blocks and spikes that only the published curve has, like b2c47 cycle 630 |
| abnormal recording | the cycle took longer than 2 h (normally about 1 h), the discharge came in more than one piece, or the discharge-capacity counter did not start at 0 | interrupted or restarted tests, like the four capacity spikes of §4 |
| spike at the 2.0 V end | the largest value of the ROI sits in its last 7.5 mV, where the discharge ends | a one-point spike far from the peak; it does not change the features, and the whole-ROI panels cut it off |
| noisy cell (per cell) | median RMS difference between the published curve and the smoothed rebuild (ENG-05's `rc_rms_diff`) | cells whose measurements are noisy throughout, like b2c40 |

The table also shows how far each group's `height` sits from the median of its ten neighbouring
cycles in the same cell, which measures how much the problem moves the features.

### 6.1 How many, and how much they matter

In [ ]:
CHECKS = {
    "inconsistent": ("published curve inconsistent", SCAN["block_points"] >= BLOCK_POINTS),
    "abnormal": ("abnormal recording", (SCAN["hours"] > LONG_RECORDING_H)
                 | (SCAN["discharge_pieces"] > 1) | (SCAN["qd_start"] > 0.05)),
    "edge_spike": ("spike at the 2.0 V end", SCAN["edge_spike"].eq(True)),
}

# How far each cycle's height sits from the median of its ten neighbouring cycles (same cell).
ok_rows = FEAT[FEAT["status"] == "ok"]
neighbours = ok_rows.groupby(level="cell_id")["height"].transform(
    lambda h: h.rolling(11, center=True, min_periods=6).median())
height_off = (ok_rows["height"] - neighbours).abs() / neighbours


def _check_row(title, keys):
    batch = keys.get_level_values("cell_id").str[:2]
    return {"check": title, "cycles": len(keys),
            "cells": keys.get_level_values("cell_id").nunique(),
            **{b: int((batch == b).sum()) for b in ("b1", "b2", "b3")},
            "kept rows": int(FEAT.loc[keys, "reason"].eq("kept").sum()),
            "height off its neighbours (median)": f"{height_off.reindex(keys).median():.1%}"}


flagged = pd.Series(False, index=SCAN.index)
rows = []
for title, mask in CHECKS.values():
    flagged |= mask
    rows.append(_check_row(title, SCAN.index[mask]))
rows.append(_check_row("none of these", SCAN.index[~flagged]))
display(pd.DataFrame(rows).set_index("check"))

inconsistent = CHECKS["inconsistent"][1]
middle_v = SCAN[["block_lo_v", "block_hi_v"]].mean(axis=1)
near = inconsistent & (middle_v - FEAT["position"].reindex(SCAN.index)).abs().lt(0.06)
first_rate = META["policy"].str.extract(r"^([0-9.]+)C")[0].astype(float)
fast = first_rate.reindex(SCAN.index[inconsistent].get_level_values("cell_id")).ge(5.6).mean()
print(f"{near.sum()} of the {inconsistent.sum()} inconsistent cycles have the problem within 60 mV "
      f"of the tracked peak; their height sits {height_off.reindex(SCAN.index[near]).median():.0%} "
      f"from their neighbours (median), against "
      f"{height_off.reindex(SCAN.index[~flagged]).median():.1%} for unflagged cycles.")
print(f"{fast:.0%} of the inconsistent cycles are in cells whose first charging step is 5.6C or "
      f"faster ({int((first_rate >= 5.6).sum())} of {len(first_rate)} cells).")

### 6.2 Which cells

Left: the cells whose published curve is noisiest against the smoothed rebuild. Right: the cells
with the most inconsistent cycles. b2c40 is in blue. Both lists are made of fast-charging cells
from batches 1 and 2.

In [ ]:
noise = ok_rows.groupby(level="cell_id")["rc_rms_diff"].median()
per_cell = inconsistent.groupby(level="cell_id").sum()
fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.4), gridspec_kw=dict(wspace=0.62))
for ax, values, title, xlabel in (
        (axes[0], noise, "noisiest cells", "median RMS difference, published vs recomputed (Ah/V)"),
        (axes[1], per_cell, "most cycles with an inconsistent published curve", "cycles")):
    top15 = values.sort_values().tail(15)
    ax.barh(np.arange(top15.size), top15.to_numpy(), height=0.62,
            color=[C1 if c == "b2c40" else AXIS for c in top15.index])
    ax.set_yticks(np.arange(top15.size), [f"{c}   {META.loc[c, 'policy']}" for c in top15.index])
    ax.tick_params(axis="y", length=0, labelcolor=INK)
    ax.grid(axis="y", visible=False)
    ax.set_title(title, loc="left", fontsize=10, fontweight="semibold")
    ax.set_xlabel(xlabel)
axes[0].axvline(noise.median(), color=INK2, lw=0.9, ls=(0, (3, 2)))
axes[0].text(noise.median(), 14.55, f" median of all {noise.size} cells", color=INK2, fontsize=7.8,
             va="center")
plt.show()

### 6.3 Browser

Choose a check, a cell and one of its flagged cycles; the figure is `show_recomputed` (§5.2).
◀ / ▶ step through that cell's flagged cycles. It starts on the batch-1 cell with the most
inconsistent cycles.

In [ ]:
def _flagged_cells(check):
    counts = CHECKS[check][1].groupby(level="cell_id").sum()
    return [(f"{c}  ({int(counts[c])} cycles)", c) for c in CELL_IDS if counts.get(c, 0) > 0]


def _flagged_cycles(check, cid):
    mask = CHECKS[check][1].loc[cid]
    return [int(k) for k in mask.index[mask]]


check_w = widgets.Dropdown(
    options=[(f"{title}  ({int(mask.sum()):,} cycles)", name)
             for name, (title, mask) in CHECKS.items()],
    value="inconsistent", description="Check", style=_wide, layout=widgets.Layout(width="440px"))
fcell_w = widgets.Dropdown(description="Cell", style=_wide, layout=widgets.Layout(width="260px"))
fcycle_w = widgets.Dropdown(description="Cycle", style=_wide, layout=widgets.Layout(width="200px"))
back3_w = widgets.Button(description="◀", layout=widgets.Layout(width="44px"))
fwd3_w = widgets.Button(description="▶", layout=widgets.Layout(width="44px"))
out3_w = widgets.Output()
_busy3 = {"on": False}


def _render_flagged(*_):
    if not _busy3["on"]:
        render_into(out3_w, lambda: show_recomputed(fcell_w.value, fcycle_w.value))


def _on_fcell(*_):
    if _busy3["on"]:
        return
    _busy3["on"] = True
    try:
        fcycle_w.options = _flagged_cycles(check_w.value, fcell_w.value)
        fcycle_w.value = fcycle_w.options[0]
    finally:
        _busy3["on"] = False
    _render_flagged()


def _on_check(*_):
    _busy3["on"] = True
    try:
        fcell_w.options = _flagged_cells(check_w.value)
        fcell_w.value = fcell_w.options[0][1]
    finally:
        _busy3["on"] = False
    _on_fcell()


def _step_flagged(delta):
    values = list(fcycle_w.options)
    i = values.index(fcycle_w.value)
    fcycle_w.value = values[int(np.clip(i + delta, 0, len(values) - 1))]


check_w.observe(_on_check, names="value")
fcell_w.observe(_on_fcell, names="value")
fcycle_w.observe(_render_flagged, names="value")
back3_w.on_click(lambda _: _step_flagged(-1))
fwd3_w.on_click(lambda _: _step_flagged(+1))

_busy3["on"] = True
fcell_w.options = _flagged_cells("inconsistent")
fcell_w.value = per_cell.filter(like="b1c").idxmax()
fcycle_w.options = _flagged_cycles("inconsistent", fcell_w.value)
fcycle_w.value = int(SCAN.loc[fcell_w.value, "block_points"].idxmax())
_busy3["on"] = False
display(widgets.HBox([check_w, fcell_w, fcycle_w, back3_w, fwd3_w]), out3_w)
_render_flagged()

### 6.4 Which cells look bad

Most odd cycles sit in a few cells, and one cell-level measure separates them cleanly: the median
change of `height` from one successful cycle to the next, as a fraction of the cell's median
height. It uses no SoH label. ENG-27 calls a cell noisy when it exceeds 5 %.

In [ ]:
NOISY_STEP = 0.05  # stages/s18_noisy_cells.py uses the same rule
ok_sorted = FEAT[FEAT["status"] == "ok"].reset_index().sort_values(["cell_id", "cycle"])
height_step = ok_sorted.groupby("cell_id")["height"].agg(
    lambda h: float(np.median(np.abs(np.diff(h.to_numpy()))) / np.median(h.to_numpy())))
NOISY_CELLS = sorted(height_step.index[height_step > NOISY_STEP], key=lambda c: (int(c[1]), int(c[3:])))

ranked = height_step.sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(14.5, 4.4))
ax.bar(np.arange(ranked.size), ranked.to_numpy() * 100, width=0.72,
       color=[C1 if c in NOISY_CELLS else AXIS for c in ranked.index])
ax.axhline(NOISY_STEP * 100, color=INK2, lw=0.9, ls=(0, (3, 2)))
ax.text(ranked.size - 1, NOISY_STEP * 100 + 0.6, f"rule: {NOISY_STEP:.0%}", ha="right",
        color=INK2, fontsize=8)
ax.set_xticks([])
ax.text(len(NOISY_CELLS) / 2 - 0.5, ranked.iloc[0] * 100 * 0.62,
        f"{len(NOISY_CELLS)} noisy cells\n(listed below)", ha="center", color=INK, fontsize=8.5)
ax.text(len(NOISY_CELLS) + 2, ranked.iloc[len(NOISY_CELLS)] * 100 + 1.5,
        f"the other {ranked.size - len(NOISY_CELLS)} cells", color=INK2, fontsize=8.5)
ax.set_xlabel("cells, ranked")
ax.grid(axis="x", visible=False)
ax.set_ylabel("typical cycle-to-cycle\nchange of height (%)")
ax.set_title("every cell, ranked: how much the published-curve height jumps from one cycle to the "
             "next", loc="left", fontsize=10, fontweight="semibold")
plt.show()

quiet = height_step.drop(NOISY_CELLS)
noisy_rows = clean["cell_id"].isin(NOISY_CELLS)
print(f"{len(NOISY_CELLS)} noisy cells: {', '.join(NOISY_CELLS)}")
print(f"their height steps: {height_step[NOISY_CELLS].min():.1%} to {height_step[NOISY_CELLS].max():.1%}; "
      f"every other cell: at most {quiet.max():.1%} ({quiet.idxmax()}), median {quiet.median():.1%}")
print(f"batches: {pd.Series([c[:2] for c in NOISY_CELLS]).value_counts().sort_index().to_dict()}; "
      f"kept rows {int(noisy_rows.sum()):,} of {len(clean):,} "
      f"({int(noisy_rows[clean['batch'].isin(['b1', 'b2'])].sum()):,} of the "
      f"{int(clean['batch'].isin(['b1', 'b2']).sum()):,} training rows)")

### 6.5 What removing them changes (ENG-27)

`stages/s18_noisy_cells.py` refits three thesis models (tuned RF, ENG-09 settings, forest seeds
0–4) once on every training cell and once without the 16 noisy cells. Folds and batch 3 stay the
same (batch 3 has no noisy cell), so each pair differs only in the training data. dev-CV is scored
on the clean validation cells in both runs. The script first reproduces the stored ENG-20 and
ENG-23 numbers. The table reads its results file.

In [ ]:
ENG27 = REPO_ROOT / "results" / "eng27_noisy_cells.json"
if ENG27.exists():
    eng27 = json.loads(ENG27.read_text(encoding="utf-8"))
    names = {"level9": "all nine level features (reference)",
             "hp_hist": "height + position with history (ENG-23 best)",
             "all63": "all 63 columns (ENG-14 combined pool)"}
    rows = []
    for model, s_ in eng27["summary"].items():
        for metric, label in (("cv_clean_rmse", "dev-CV RMSE, clean cells"),
                              ("b3_rmse", "batch-3 RMSE"), ("b3_r2", "batch-3 R²")):
            a, b, c = s_["all_dev"][metric], s_["without_noisy"][metric], s_[f"change_{metric}"]
            rows.append({"model": names[model], "score": label,
                         "all training cells": f"{a['mean']:.5f} ± {a['sd']:.5f}",
                         "without the 16": f"{b['mean']:.5f} ± {b['sd']:.5f}",
                         "change": f"{c['relative_mean']:+.1%}",
                         "seeds lower": f"{c['seeds_lower']}/{c['n_seeds']}"})
    display(pd.DataFrame(rows).set_index(["model", "score"]))
    print("reproduces ENG-20 / ENG-23:", "yes" if eng27["all_checks_pass"] else "NO")
else:
    print("results/eng27_noisy_cells.json not found: run stages/s18_noisy_cells.py (~25 min)")

## 7. Notes

- **Inputs only.** The notebook reads `results/features_expanded.parquet` and the pickles. It
  writes only `dropped_rows.csv` and, if you press *Save PNG*, images in `views/` (PNG files are
  gitignored).
- **Not drawn.** The second-peak columns (`height2`, `position2`, `prom2`, ...) and the harmonic
  features are not part of the modelling pools, so they are left out.
- **The label offset (§4.2)** is a property of the extractors, not of this notebook. Fixing it means
  pairing cycle k with summary row k (`summary_cycles == k + 1`) in both extractors and then
  re-running ENG-05 and everything after it. It would keep the 124 label-dropped rows, move the
  four spike drops onto the abnormal cycles themselves, and shift every label by one cycle.
- **The odd cycles (§6)** come from the published data, not from the extractor, and the cleaning
  step keeps them, because none of its four filters looks at the curve. If they matter for the
  thesis, the options are to drop the flagged cycles or to measure the features on the recomputed
  curve (§5); either means re-running ENG-05 and everything after it.